## Mercado Libre

In [1]:
import os
import re
import time
import random
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [6]:
CARPETA_IMAGENES = "laptop_images"
if not os.path.exists(CARPETA_IMAGENES):
    os.makedirs(CARPETA_IMAGENES)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'es-MX,es;q=0.9'
}

def limpiar_precio(precio_str):
    if not precio_str:
        return None
    limpio = re.sub(r"[^\d.]", "", precio_str.replace(",", ""))
    try:
        return float(limpio)
    except ValueError:
        return None

def extraer_ram(titulo):
    match = re.search(r"(\d+)\s*(gb|mb)\s*(?:ram|memoria)?", titulo, re.IGNORECASE)
    if match:
        valor, unidad = int(match.group(1)), match.group(2).lower()
        return valor if unidad == "gb" else int(valor / 1024)
    return None

def extraer_almacenamiento(titulo):
    match = re.search(r"(\d+)\s*(tb|gb)\s*(ssd|hdd|nvme)?|(ssd|hdd|nvme)\s*(\d+)\s*(tb|gb)", titulo, re.IGNORECASE)
    if match:
        groups = [g for g in match.groups() if g is not None]
        num = next((int(g) for g in groups if g.isdigit()), None)
        unit = next((g.lower() for g in groups if g.lower() in ["gb", "tb"]), "gb")
        tipo = next((g.upper() for g in groups if g.upper() in ["SSD", "HDD", "NVME"]), "SSD")
        if num:
            capacidad_gb = num * 1024 if unit == "tb" else num
            return capacidad_gb, tipo
    return None, "SSD"

def extraer_marca(titulo):
    marcas_patron = r"(HP|Lenovo|Asus|Dell|Apple|MacBook|Acer|MSI|Huawei|Samsung|Gigabyte|Machcreator|Alienware)"
    match = re.search(marcas_patron, titulo, re.IGNORECASE)
    if match:
        marca = match.group(1)
        return "Apple" if marca.lower() == "macbook" else marca.capitalize()
    return "Otra"

def extraer_procesador(titulo):
    match = re.search(r"(Core\s*i\d|Ryzen\s*\d|Apple\s*M\d|M\d\s*Max|M\d\s*Pro|Intel\s*N\d+|Celeron|Athlon)", titulo, re.IGNORECASE)
    return match.group(1).title() if match else "No especificado"

def extraer_pantalla(titulo):
    match = re.search(r'(\d{2}(?:\.\d)?)\s*(?:"|pulgadas|”)', titulo, re.IGNORECASE)
    return float(match.group(1)) if match else None

def detectar_gpu_dedicada(titulo):
    gpus = ["rtx", "gtx", "radeon rx", "quadro", "geforce mx", "arc a"]
    return 1 if any(gpu in titulo.lower() for gpu in gpus) else 0

def scrapear_mercadolibre_masivo(max_productos=5000):
    laptops_lista = []
    urls_vistas = set()
    idx = 0
    pagina = 0
    paginas_sin_nuevos = 0

    print(f"Iniciando extracción masiva (Objetivo máximo: {max_productos} productos)...\n")

    try:
        while idx < max_productos:
            if pagina == 0:
                url_actual = 'https://listado.mercadolibre.com.mx/laptops'
            else:
                offset = (pagina * 48) + 1
                url_actual = f'https://listado.mercadolibre.com.mx/laptops_Desde_{offset}_NoIndex_True'
            
            print(f"-> Escaneando página {pagina + 1} | Registros acumulados: {idx}")
            
            response = requests.get(url_actual, headers=HEADERS, timeout=20)
            
            if response.status_code != 200 or "ingresa a tu cuenta" in response.text.lower():
                print("\n[!] Alerta de seguridad o bloqueo detectado. Finalizando recolección...")
                break
                
            soup = BeautifulSoup(response.text, "html.parser")
            tarjetas = soup.find_all(["div", "li"], class_=re.compile(r"poly-card|ui-search-layout__item"))

            if not tarjetas:
                print("\n[!] No se encontraron más publicaciones en la página. Fin del catálogo.")
                break

            nuevos_en_esta_pagina = 0

            for item in tarjetas:
                if idx >= max_productos:
                    break

                titulo_elem = item.find("a", class_=re.compile(r"poly-component__title|ui-search-item__title")) or item.find(re.compile(r"h[2-3]"))
                titulo = titulo_elem.text.strip() if titulo_elem else ""

                if not titulo or titulo == "Sin título" or "mac mini" in titulo.lower():
                    continue

                url_producto = titulo_elem["href"] if titulo_elem and titulo_elem.has_attr("href") else ""
                
                # Descartar publicaciones repetidas (MercadoLibre reinicia la paginación al final)
                if not url_producto or url_producto in urls_vistas:
                    continue
                
                urls_vistas.add(url_producto)

                precio_elem = item.find("span", class_="andes-money-amount__fraction")
                precio_num = limpiar_precio(precio_elem.text.strip()) if precio_elem else None

                if not precio_num or precio_num == 0:
                    continue
                
                img_elem = item.find("img")
                img_url = img_elem.get("data-src") or img_elem.get("src") or "" if img_elem else ""

                ruta_img_local = ""
                if img_url and img_url.startswith("http"):
                    ruta_img_local = os.path.join(CARPETA_IMAGENES, f"laptop_{idx:04d}.jpg")
                    try:
                        img_data = requests.get(img_url, timeout=3).content
                        with open(ruta_img_local, "wb") as f:
                            f.write(img_data)
                    except Exception:
                        ruta_img_local = ""

                laptops_lista.append({
                    "ID_Producto": f"LAP-{idx:04d}",
                    "Nombre": titulo,
                    "Marca": extraer_marca(titulo),
                    "Precio_MXN": precio_num,
                    "RAM_GB": extraer_ram(titulo),
                    "Almacenamiento_GB": extraer_almacenamiento(titulo)[0],
                    "Tipo_Almacenamiento": extraer_almacenamiento(titulo)[1],
                    "Procesador": extraer_procesador(titulo),
                    "Pantalla_Pulgadas": extraer_pantalla(titulo),
                    "GPU_Dedicada": detectar_gpu_dedicada(titulo),
                    "Ruta_Imagen_Local": ruta_img_local,
                    "URL_Producto": url_producto,
                })
                
                idx += 1
                nuevos_en_esta_pagina += 1

            # Control de fin de catálogo real
            if nuevos_en_esta_pagina == 0:
                paginas_sin_nuevos += 1
                if paginas_sin_nuevos >= 2:
                    print("\n[!] Se detectó el final del catálogo (2 páginas seguidas sin productos nuevos).")
                    break
            else:
                paginas_sin_nuevos = 0

            # Guardado preventivo cada 500 registros
            if idx > 0 and idx % 500 == 0:
                pd.DataFrame(laptops_lista).to_csv("laptops_base_explicita.csv", index=False, encoding='utf-8')
                print(f"   --> [Autoguardado] {idx} registros respaldados en CSV.")

            pagina += 1
            time.sleep(random.uniform(1.8, 3.0))

    except KeyboardInterrupt:
        print("\n[!] Extracción detenida manualmente.")
    except Exception as e:
        print(f"\n[!] Ocurrió una interrupción: {e}")
    finally:
        if laptops_lista:
            df_resultado = pd.DataFrame(laptops_lista)
            df_resultado.to_csv("laptops_base_ml_masiva_2.csv", index=False, encoding='utf-8')
            print(f"\n¡Proceso finalizado! Se guardaron exitosamente {len(laptops_lista)} registros finales en 'laptops_base_explicita.csv'.")
            return df_resultado
        else:
            print("\nNo se obtuvo ningún registro.")
            return pd.DataFrame()

if __name__ == "__main__":
    # Ejecuta intentando llegar a 5000; si hay menos (ej. 2500 o 3100), los guardará todos sin error.
    df_final = scrapear_mercadolibre_masivo(max_productos=5000)

Iniciando extracción masiva (Objetivo máximo: 5000 productos)...

-> Escaneando página 1 | Registros acumulados: 0
-> Escaneando página 2 | Registros acumulados: 47
-> Escaneando página 3 | Registros acumulados: 94
-> Escaneando página 4 | Registros acumulados: 141
-> Escaneando página 5 | Registros acumulados: 188
-> Escaneando página 6 | Registros acumulados: 235
-> Escaneando página 7 | Registros acumulados: 282
-> Escaneando página 8 | Registros acumulados: 329
-> Escaneando página 9 | Registros acumulados: 376
-> Escaneando página 10 | Registros acumulados: 423
-> Escaneando página 11 | Registros acumulados: 470
-> Escaneando página 12 | Registros acumulados: 517
-> Escaneando página 13 | Registros acumulados: 564
-> Escaneando página 14 | Registros acumulados: 611
-> Escaneando página 15 | Registros acumulados: 658
-> Escaneando página 16 | Registros acumulados: 705
-> Escaneando página 17 | Registros acumulados: 752
-> Escaneando página 18 | Registros acumulados: 799
-> Escanean

### Elecktra

In [4]:
import re
import time
import random
import pandas as pd
import requests

# --- Funciones de Extracción Regex ---
def extraer_ram(titulo):
    match = re.search(r"(\d+)\s*(gb|mb)\s*(?:ddr\d|ram|memoria)?", titulo, re.IGNORECASE)
    if match: 
        return int(match.group(1)) if match.group(2).lower() == "gb" else int(int(match.group(1)) / 1024)
    return None

def extraer_almacenamiento(titulo):
    match = re.search(r"(\d+)\s*(tb|gb)\s*(ssd|hdd|nvme)?|(ssd|hdd|nvme)\s*(\d+)\s*(tb|gb)", titulo, re.IGNORECASE)
    if match:
        groups = [g for g in match.groups() if g is not None]
        num = next((int(g) for g in groups if g.isdigit()), None)
        unit = next((g.lower() for g in groups if g.lower() in ["gb", "tb"]), "gb")
        tipo = next((g.upper() for g in groups if g.upper() in ["SSD", "HDD", "NVME"]), "SSD")
        if num: 
            return (num * 1024 if unit == "tb" else num), tipo
    return None, "SSD"

def extraer_marca(titulo, marca_vtex=""):
    if marca_vtex and marca_vtex.lower() not in ["elektra", "generica", "sin marca"]:
        return marca_vtex.capitalize()
    marcas = r"(HP|Lenovo|Asus|Dell|Apple|MacBook|Acer|MSI|Huawei|Samsung|Gigabyte|Alienware)"
    match = re.search(marcas, titulo, re.IGNORECASE)
    if match:
        m = match.group(1).capitalize()
        return "Apple" if m.lower() == "macbook" else m
    return "Otra"

def extraer_procesador(titulo):
    match = re.search(r"(Core\s*i\d|Core\s*\d|Ryzen\s*\d|Apple\s*M\d|Intel\s*N\d+|Celeron|Athlon)", titulo, re.IGNORECASE)
    return match.group(1).title() if match else "No especificado"

def extraer_pantalla(titulo):
    match = re.search(r'(\d{2}(?:\.\d)?)\s*(?:"|pulgadas|”)', titulo, re.IGNORECASE)
    return float(match.group(1)) if match else None

def detectar_gpu_dedicada(titulo):
    return 1 if any(gpu in titulo.lower() for gpu in ["rtx", "gtx", "radeon rx", "quadro", "arc a"]) else 0

# --- Extractor con Control de Seguridad y Escala ---
def scraper_elektra(max_productos=5000):
    print(f"\n--- Extrayendo catálogo de Elektra México (Objetivo: {max_productos}) ---")
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'application/json'
    }
    
    laptops = []
    urls_vistas = set()
    idx = 0
    paso = 50
    desde = 0
    sin_nuevos = 0

    try:
        while idx < max_productos:
            hasta = desde + paso - 1
            url_api = f"https://www.elektra.mx/api/catalog_system/pub/products/search/computo/portatiles/laptops?_from={desde}&_to={hasta}"
            print(f"-> Consultando registros {desde} a {hasta} | Total acumulado: {idx}")
            
            res = requests.get(url_api, headers=headers, timeout=15)
            
            # Detener si hay un código HTTP inesperado (bloqueo 403, 429, etc.)
            if res.status_code not in [200, 206]:
                print(f"\n[!] Se detectó una respuesta inusual (Código HTTP {res.status_code}). Deteniendo para respaldar datos.")
                break
                
            productos = res.json()
            if not productos:
                print("\n[!] Fin del catálogo alcanzado (no hay más productos).")
                break

            nuevos_en_lote = 0

            for prod in productos:
                if idx >= max_productos: 
                    break
                
                url_prod = prod.get("link", "")
                
                # Evitar registros duplicados
                if url_prod in urls_vistas:
                    continue
                urls_vistas.add(url_prod)

                titulo = prod.get("productName", "").strip()
                marca_raw = prod.get("brand", "")
                
                precio = None
                items = prod.get("items", [])
                if items:
                    sellers = items[0].get("sellers", [])
                    if sellers:
                        offer = sellers[0].get("commertialOffer", {})
                        precio = offer.get("Price") or offer.get("ListPrice")

                if not titulo or not precio or precio <= 0: 
                    continue

                alm_gb, alm_tipo = extraer_almacenamiento(titulo)
                
                laptops.append({
                    "ID_Producto": f"EKT-{idx:04d}",
                    "Marca": extraer_marca(titulo, marca_raw),
                    "Nombre": titulo,
                    "Precio_MXN": float(precio),
                    "RAM_GB": extraer_ram(titulo),
                    "Almacenamiento_GB": alm_gb,
                    "Tipo_Almacenamiento": alm_tipo,
                    "Procesador": extraer_procesador(titulo),
                    "GPU_Dedicada": detectar_gpu_dedicada(titulo),
                    "Pantalla_Pulgadas": extraer_pantalla(titulo),
                    "URL_Producto": url_prod
                })
                idx += 1
                nuevos_en_lote += 1

            # Detectar si la API comenzó a repetir resultados
            if nuevos_en_lote == 0:
                sin_nuevos += 1
                if sin_nuevos >= 2:
                    print("\n[!] Se alcanzó el límite del catálogo disponible (sin productos nuevos).")
                    break
            else:
                sin_nuevos = 0

            # Respaldar datos automáticamente cada 500 registros
            if idx > 0 and idx % 500 == 0:
                pd.DataFrame(laptops).to_csv("laptops_elektra.csv", index=False, encoding='utf-8')
                print(f"   --> [Autoguardado] {idx} registros asegurados en 'laptops_elektra.csv'.")

            desde += paso
            time.sleep(random.uniform(1.2, 2.5))

    except KeyboardInterrupt:
        print("\n[!] Proceso detenido manualmente.")
    except Exception as e:
        print(f"\n[!] Error inesperado: {e}")
    finally:
        # Garantiza el guardado de todo lo recopilado hasta el momento de cierre o fallo
        if laptops:
            df_final = pd.DataFrame(laptops)
            df_final.to_csv("laptops_elektra_2.csv", index=False, encoding='utf-8')
            print(f"\n¡Proceso finalizado! Se guardaron {len(df_final)} registros en 'laptops_elektra.csv'.")
            return df_final
        else:
            print("\nNo se recopilaron datos.")
            return pd.DataFrame()

if __name__ == "__main__":
    df = scraper_elektra(max_productos=5000)


--- Extrayendo catálogo de Elektra México (Objetivo: 5000) ---
-> Consultando registros 0 a 49 | Total acumulado: 0
-> Consultando registros 50 a 99 | Total acumulado: 50
-> Consultando registros 100 a 149 | Total acumulado: 99
-> Consultando registros 150 a 199 | Total acumulado: 149
-> Consultando registros 200 a 249 | Total acumulado: 199
-> Consultando registros 250 a 299 | Total acumulado: 249
-> Consultando registros 300 a 349 | Total acumulado: 298
-> Consultando registros 350 a 399 | Total acumulado: 348
-> Consultando registros 400 a 449 | Total acumulado: 398
-> Consultando registros 450 a 499 | Total acumulado: 448
-> Consultando registros 500 a 549 | Total acumulado: 498
-> Consultando registros 550 a 599 | Total acumulado: 547
-> Consultando registros 600 a 649 | Total acumulado: 597
-> Consultando registros 650 a 699 | Total acumulado: 647
-> Consultando registros 700 a 749 | Total acumulado: 696
-> Consultando registros 750 a 799 | Total acumulado: 745
-> Consultando re

## Chedraui

In [14]:
import re
import time
import random
import pandas as pd
import requests

# --- Funciones de Extracción Regex ---
def extraer_ram(titulo):
    match = re.search(r"(\d+)\s*(gb|mb)\s*(?:ddr\d|ram|memoria)?", titulo, re.IGNORECASE)
    if match: 
        return int(match.group(1)) if match.group(2).lower() == "gb" else int(int(match.group(1)) / 1024)
    return None

def extraer_almacenamiento(titulo):
    match = re.search(r"(\d+)\s*(tb|gb)\s*(ssd|hdd|nvme)?|(ssd|hdd|nvme)\s*(\d+)\s*(tb|gb)", titulo, re.IGNORECASE)
    if match:
        groups = [g for g in match.groups() if g is not None]
        num = next((int(g) for g in groups if g.isdigit()), None)
        unit = next((g.lower() for g in groups if g.lower() in ["gb", "tb"]), "gb")
        tipo = next((g.upper() for g in groups if g.upper() in ["SSD", "HDD", "NVME"]), "SSD")
        if num: 
            return (num * 1024 if unit == "tb" else num), tipo
    return None, "SSD"

def extraer_marca(titulo, marca_vtex=""):
    if marca_vtex and marca_vtex.lower() not in ["chedraui", "generica", "sin marca"]:
        return marca_vtex.capitalize()
    marcas = r"(HP|Lenovo|Asus|Dell|Apple|MacBook|Acer|MSI|Huawei|Samsung|Gigabyte|Alienware|Lanix)"
    match = re.search(marcas, titulo, re.IGNORECASE)
    if match:
        m = match.group(1).capitalize()
        return "Apple" if m.lower() == "macbook" else m
    return "Otra"

def extraer_procesador(titulo):
    match = re.search(r"(Core\s*i\d|Core\s*\d|Ryzen\s*\d|Apple\s*M\d|Intel\s*N\d+|Celeron|Athlon)", titulo, re.IGNORECASE)
    return match.group(1).title() if match else "No especificado"

def extraer_pantalla(titulo):
    match = re.search(r'(\d{2}(?:\.\d)?)\s*(?:"|pulgadas|”)', titulo, re.IGNORECASE)
    return float(match.group(1)) if match else None

def detectar_gpu_dedicada(titulo):
    return 1 if any(gpu in titulo.lower() for gpu in ["rtx", "gtx", "radeon rx", "quadro", "arc a"]) else 0

# --- Extractor Chedraui México (VTEX API) ---
def scraper_chedraui(max_productos=3000):
    print(f"\n--- Extrayendo catálogo de Chedraui México (Objetivo: {max_productos}) ---")
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'application/json',
        'Accept-Language': 'es-MX,es;q=0.9,en-US;q=0.8,en;q=0.7'
    }
    
    laptops = []
    urls_vistas = set()
    idx = 0
    paso = 50
    desde = 0
    sin_nuevos = 0

    try:
        while idx < max_productos:
            hasta = desde + paso - 1
            # Consulta directa mediante búsqueda global 'laptop' en API VTEX
            url_api = f"https://www.chedraui.com.mx/api/catalog_system/pub/products/search?ft=laptop&_from={desde}&_to={hasta}"
            print(f"-> Consultando registros {desde} a {hasta} | Total acumulado: {idx}")
            
            res = requests.get(url_api, headers=headers, timeout=15)
            
            # 200 y 206 son respuestas válidas en VTEX para entregas parciales
            if res.status_code not in [200, 206]:
                print(f"\n[!] HTTP {res.status_code}. Deteniendo extracción y respaldando acumulado...")
                break
                
            try:
                productos = res.json()
            except Exception:
                print("\n[!] Fin de datos o la API no entregó JSON en este rango.")
                break

            if not productos or not isinstance(productos, list):
                print("\n[!] No se encontraron más productos en la respuesta.")
                break

            nuevos_en_lote = 0

            for prod in productos:
                if idx >= max_productos: break
                
                url_prod = prod.get("link", "")
                if not url_prod or url_prod in urls_vistas:
                    continue
                urls_vistas.add(url_prod)

                titulo = prod.get("productName", "").strip()
                marca_vtex = prod.get("brand", "")
                
                # Obtener precio comercial actualizado
                precio = None
                items = prod.get("items", [])
                if items:
                    sellers = items[0].get("sellers", [])
                    if sellers:
                        offer = sellers[0].get("commertialOffer", {})
                        precio = offer.get("Price") or offer.get("ListPrice")

                if not titulo or not precio or precio <= 0:
                    continue

                alm_gb, alm_tipo = extraer_almacenamiento(titulo)
                
                laptops.append({
                    "ID_Producto": f"CHE-{idx:04d}",
                    "Marca": extraer_marca(titulo, marca_vtex),
                    "Nombre": titulo,
                    "Precio_MXN": float(precio),
                    "RAM_GB": extraer_ram(titulo),
                    "Almacenamiento_GB": alm_gb,
                    "Tipo_Almacenamiento": alm_tipo,
                    "Procesador": extraer_procesador(titulo),
                    "GPU_Dedicada": detectar_gpu_dedicada(titulo),
                    "Pantalla_Pulgadas": extraer_pantalla(titulo),
                    "URL_Producto": url_prod
                })
                idx += 1
                nuevos_en_lote += 1

            if nuevos_en_lote == 0:
                sin_nuevos += 1
                if sin_nuevos >= 2:
                    print("\n[!] Fin del catálogo alcanzado.")
                    break
            else:
                sin_nuevos = 0

            # Resguardo continuo cada 300 productos
            if idx > 0 and idx % 300 == 0:
                pd.DataFrame(laptops).to_csv("laptops_chedraui.csv", index=False, encoding='utf-8')
                print(f"   --> [Autoguardado] {idx} registros asegurados en 'laptops_chedraui.csv'.")

            desde += paso
            time.sleep(random.uniform(1.0, 2.0))

    except KeyboardInterrupt:
        print("\n[!] Proceso detenido manualmente.")
    except Exception as e:
        print(f"\n[!] Error durante la ejecución: {e}")
    finally:
        if laptops:
            df_final = pd.DataFrame(laptops)
            df_final.to_csv("laptops_chedraui.csv", index=False, encoding='utf-8')
            print(f"\n¡Proceso finalizado! Se guardaron {len(df_final)} registros en 'laptops_chedraui.csv'.")
            return df_final
        else:
            print("\nNo se recopilaron datos.")
            return pd.DataFrame()

if __name__ == "__main__":
    df = scraper_chedraui(max_productos=3000)


--- Extrayendo catálogo de Chedraui México (Objetivo: 3000) ---
-> Consultando registros 0 a 49 | Total acumulado: 0
-> Consultando registros 50 a 99 | Total acumulado: 50
-> Consultando registros 100 a 149 | Total acumulado: 99
-> Consultando registros 150 a 199 | Total acumulado: 106

[!] No se encontraron más productos en la respuesta.

¡Proceso finalizado! Se guardaron 106 registros en 'laptops_chedraui.csv'.


## Mercado Libre - Por categoria

In [19]:
import os
import re
import time
import random
import pandas as pd
import requests
from bs4 import BeautifulSoup

# --- Configuración ---
CARPETA_IMAGENES = "laptop_images"
ARCHIVO_CSV = "laptops_mercadolibre.csv"

if not os.path.exists(CARPETA_IMAGENES):
    os.makedirs(CARPETA_IMAGENES)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'es-MX,es;q=0.9'
}

# --- Helper Functions ---
def limpiar_precio(precio_str):
    if not precio_str: return None
    limpio = re.sub(r"[^\d.]", "", precio_str.replace(",", ""))
    try: return float(limpio)
    except ValueError: return None

def extraer_ram(titulo):
    match = re.search(r"(\d+)\s*(gb|mb)\s*(?:ram|memoria)?", titulo, re.IGNORECASE)
    if match:
        valor, unidad = int(match.group(1)), match.group(2).lower()
        return valor if unidad == "gb" else int(valor / 1024)
    return None

def extraer_almacenamiento(titulo):
    match = re.search(r"(\d+)\s*(tb|gb)\s*(ssd|hdd|nvme)?|(ssd|hdd|nvme)\s*(\d+)\s*(tb|gb)", titulo, re.IGNORECASE)
    if match:
        groups = [g for g in match.groups() if g is not None]
        num = next((int(g) for g in groups if g.isdigit()), None)
        unit = next((g.lower() for g in groups if g.lower() in ["gb", "tb"]), "gb")
        tipo = next((g.upper() for g in groups if g.upper() in ["SSD", "HDD", "NVME"]), "SSD")
        if num:
            return (num * 1024 if unit == "tb" else num), tipo
    return None, "SSD"

def extraer_marca(titulo):
    marcas_patron = r"(HP|Lenovo|Asus|Dell|Apple|MacBook|Acer|MSI|Huawei|Samsung|Gigabyte|Machcreator|Alienware)"
    match = re.search(marcas_patron, titulo, re.IGNORECASE)
    if match:
        marca = match.group(1)
        return "Apple" if marca.lower() == "macbook" else marca.capitalize()
    return "Otra"

def extraer_procesador(titulo):
    match = re.search(r"(Core\s*i\d|Ryzen\s*\d|Apple\s*M\d|M\d\s*Max|M\d\s*Pro|Intel\s*N\d+|Celeron|Athlon)", titulo, re.IGNORECASE)
    return match.group(1).title() if match else "No especificado"

def extraer_pantalla(titulo):
    match = re.search(r'(\d{2}(?:\.\d)?)\s*(?:"|pulgadas|”)', titulo, re.IGNORECASE)
    return float(match.group(1)) if match else None

def detectar_gpu_dedicada(titulo):
    gpus = ["rtx", "gtx", "radeon rx", "quadro", "geforce mx", "arc a"]
    return 1 if any(gpu in titulo.lower() for gpu in gpus) else 0

# --- Función Principal con Reanudación ---
def reanudar_y_scrapear_masivo(max_productos=5000):
    laptops_lista = []
    urls_vistas = set()
    idx = 0

    # Cargar avance previo si existe el CSV
    if os.path.exists(ARCHIVO_CSV):
        try:
            df_previo = pd.read_csv(ARCHIVO_CSV)
            laptops_lista = df_previo.to_dict(orient="records")
            urls_vistas = set(df_previo["URL_Producto"].dropna().tolist())
            idx = len(laptops_lista)
            print(f"[i] Avance detectado: Se cargaron {idx} registros previos. Reanudando desde 'idx={idx}'...\n")
        except Exception as e:
            print(f"[!] No se pudo leer el archivo existente ({e}). Iniciando desde cero.\n")

    # Búsquedas segmentadas para evade la restricción de ~2000 ítems por listado
    segmentos = [
        "https://listado.mercadolibre.com.mx/laptop-hp",
        "https://listado.mercadolibre.com.mx/laptop-lenovo",
        "https://listado.mercadolibre.com.mx/laptop-gamer",
        "https://listado.mercadolibre.com.mx/laptop-dell",
        "https://listado.mercadolibre.com.mx/laptop-asus",
        "https://listado.mercadolibre.com.mx/laptop-acer",
        "https://listado.mercadolibre.com.mx/macbook",
        "https://listado.mercadolibre.com.mx/laptop-huawei",
        "https://listado.mercadolibre.com.mx/laptop-16gb-ram"
    ]

    try:
        for url_base in segmentos:
            if idx >= max_productos:
                break
            
            print(f"\n=== Iniciando segmento: {url_base.split('/')[-1]} ===")
            pagina = 0
            paginas_sin_nuevos = 0

            while idx < max_productos:
                if pagina == 0:
                    url_actual = url_base
                else:
                    offset = (pagina * 48) + 1
                    slug = url_base.split('/')[-1]
                    url_actual = f'https://listado.mercadolibre.com.mx/{slug}_Desde_{offset}_NoIndex_True'

                print(f"-> Escaneando página {pagina + 1} | Registros acumulados: {idx}")
                
                response = requests.get(url_actual, headers=HEADERS, timeout=20)
                
                if response.status_code != 200 or "ingresa a tu cuenta" in response.text.lower():
                    print(f"  [!] Límite de paginación o HTTP {response.status_code} alcanzado para este segmento.")
                    break
                    
                soup = BeautifulSoup(response.text, "html.parser")
                tarjetas = soup.find_all(["div", "li"], class_=re.compile(r"poly-card|ui-search-layout__item"))

                if not tarjetas:
                    print("  [!] Fin de resultados para este segmento.")
                    break

                nuevos_en_esta_pagina = 0

                for item in tarjetas:
                    if idx >= max_productos:
                        break

                    titulo_elem = item.find("a", class_=re.compile(r"poly-component__title|ui-search-item__title")) or item.find(re.compile(r"h[2-3]"))
                    titulo = titulo_elem.text.strip() if titulo_elem else ""

                    if not titulo or titulo == "Sin título" or "mac mini" in titulo.lower():
                        continue

                    url_producto = titulo_elem["href"] if titulo_elem and titulo_elem.has_attr("href") else ""
                    
                    # Evitar procesar registros duplicados entre segmentos
                    if not url_producto or url_producto in urls_vistas:
                        continue
                    
                    urls_vistas.add(url_producto)

                    precio_elem = item.find("span", class_="andes-money-amount__fraction")
                    precio_num = limpiar_precio(precio_elem.text.strip()) if precio_elem else None

                    if not precio_num or precio_num == 0:
                        continue
                    
                    # Descargar imagen local
                    img_elem = item.find("img")
                    img_url = img_elem.get("data-src") or img_elem.get("src") or "" if img_elem else ""

                    ruta_img_local = ""
                    if img_url and img_url.startswith("http"):
                        ruta_img_local = os.path.join(CARPETA_IMAGENES, f"laptop_{idx:04d}.jpg")
                        try:
                            img_data = requests.get(img_url, headers=HEADERS, timeout=5).content
                            with open(ruta_img_local, "wb") as f:
                                f.write(img_data)
                        except Exception:
                            ruta_img_local = ""

                    alm_cap, alm_tipo = extraer_almacenamiento(titulo)

                    laptops_lista.append({
                        "ID_Producto": f"MLM-{idx:04d}",
                        "Nombre": titulo,
                        "Marca": extraer_marca(titulo),
                        "Precio_MXN": precio_num,
                        "RAM_GB": extraer_ram(titulo),
                        "Almacenamiento_GB": alm_cap,
                        "Tipo_Almacenamiento": alm_tipo,
                        "Procesador": extraer_procesador(titulo),
                        "Pantalla_Pulgadas": extraer_pantalla(titulo),
                        "GPU_Dedicada": detectar_gpu_dedicada(titulo),
                        "Ruta_Imagen_Local": ruta_img_local,
                        "URL_Producto": url_producto,
                    })
                    
                    idx += 1
                    nuevos_en_esta_pagina += 1

                if nuevos_en_esta_pagina == 0:
                    paginas_sin_nuevos += 1
                    if paginas_sin_nuevos >= 2:
                        print("  [!] 2 páginas consecutivas sin productos nuevos. Cambiando de segmento.")
                        break
                else:
                    paginas_sin_nuevos = 0

                # Resguardo periódico
                if idx > 0 and idx % 200 == 0:
                    pd.DataFrame(laptops_lista).to_csv(ARCHIVO_CSV, index=False, encoding='utf-8')
                    print(f"   --> [Autoguardado] {idx} registros respaldados en CSV.")

                pagina += 1
                time.sleep(random.uniform(1.5, 2.8))

    except KeyboardInterrupt:
        print("\n[!] Proceso detenido manualmente.")
    except Exception as e:
        print(f"\n[!] Excepción ocurrida: {e}")
    finally:
        if laptops_lista:
            df_final = pd.DataFrame(laptops_lista)
            df_final.to_csv(ARCHIVO_CSV, index=False, encoding='utf-8')
            print(f"\n¡Proceso finalizado! Se guardaron {len(df_final)} registros en '{ARCHIVO_CSV}'.")
            return df_final
        else:
            print("\nNo se registraron datos.")
            return pd.DataFrame()

if __name__ == "__main__":
    df = reanudar_y_scrapear_masivo(max_productos=10000)

[i] Avance detectado: Se cargaron 1974 registros previos. Reanudando desde 'idx=1974'...


=== Iniciando segmento: laptop-hp ===
-> Escaneando página 1 | Registros acumulados: 1974
-> Escaneando página 2 | Registros acumulados: 2022
-> Escaneando página 3 | Registros acumulados: 2070
-> Escaneando página 4 | Registros acumulados: 2118
-> Escaneando página 5 | Registros acumulados: 2166
-> Escaneando página 6 | Registros acumulados: 2214
-> Escaneando página 7 | Registros acumulados: 2262
-> Escaneando página 8 | Registros acumulados: 2310
-> Escaneando página 9 | Registros acumulados: 2358
-> Escaneando página 10 | Registros acumulados: 2406
-> Escaneando página 11 | Registros acumulados: 2454
-> Escaneando página 12 | Registros acumulados: 2502
-> Escaneando página 13 | Registros acumulados: 2550
-> Escaneando página 14 | Registros acumulados: 2598
-> Escaneando página 15 | Registros acumulados: 2646
-> Escaneando página 16 | Registros acumulados: 2694
-> Escaneando página 17 | Regist

## Mercado Libre - Por categoria sin limite en el número de registros

In [1]:
import os
import re
import time
import random
import pandas as pd
import requests
from bs4 import BeautifulSoup

# --- Configuración de Archivos y Entorno ---
CARPETA_IMAGENES = "/media/solaris/017116de-6c2d-44fb-a97a-0afccc5d6037/solaris/claralap/laptop_images"
ARCHIVO_CSV = "/media/solaris/017116de-6c2d-44fb-a97a-0afccc5d6037/solaris/claralap/laptops_mercadolibre.csv"

if not os.path.exists(CARPETA_IMAGENES):
    os.makedirs(CARPETA_IMAGENES)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'es-MX,es;q=0.9'
}

# --- Funciones de Extracción Regex ---
def limpiar_precio(precio_str):
    if not precio_str: return None
    limpio = re.sub(r"[^\d.]", "", precio_str.replace(",", ""))
    try: return float(limpio)
    except ValueError: return None

def extraer_ram(titulo):
    match = re.search(r"(\d+)\s*(gb|mb)\s*(?:ram|memoria)?", titulo, re.IGNORECASE)
    if match:
        valor, unidad = int(match.group(1)), match.group(2).lower()
        return valor if unidad == "gb" else int(valor / 1024)
    return None

def extraer_almacenamiento(titulo):
    match = re.search(r"(\d+)\s*(tb|gb)\s*(ssd|hdd|nvme)?|(ssd|hdd|nvme)\s*(\d+)\s*(tb|gb)", titulo, re.IGNORECASE)
    if match:
        groups = [g for g in match.groups() if g is not None]
        num = next((int(g) for g in groups if g.isdigit()), None)
        unit = next((g.lower() for g in groups if g.lower() in ["gb", "tb"]), "gb")
        tipo = next((g.upper() for g in groups if g.upper() in ["SSD", "HDD", "NVME"]), "SSD")
        if num:
            return (num * 1024 if unit == "tb" else num), tipo
    return None, "SSD"

def extraer_marca(titulo):
    marcas_patron = r"(HP|Lenovo|Asus|Dell|Apple|MacBook|Acer|MSI|Huawei|Samsung|Gigabyte|Machcreator|Alienware|Honor|Razer|Gateway|Ghia)"
    match = re.search(marcas_patron, titulo, re.IGNORECASE)
    if match:
        marca = match.group(1)
        return "Apple" if marca.lower() == "macbook" else marca.capitalize()
    return "Otra"

def extraer_procesador(titulo):
    match = re.search(r"(Core\s*i\d|Ryzen\s*\d|Apple\s*M\d|M\d\s*Max|M\d\s*Pro|Intel\s*N\d+|Celeron|Athlon|Pentium|Snapdragon)", titulo, re.IGNORECASE)
    return match.group(1).title() if match else "No especificado"

def extraer_pantalla(titulo):
    match = re.search(r'(\d{2}(?:\.\d)?)\s*(?:"|pulgadas|”)', titulo, re.IGNORECASE)
    return float(match.group(1)) if match else None

def detectar_gpu_dedicada(titulo):
    gpus = ["rtx", "gtx", "radeon rx", "quadro", "geforce mx", "arc a"]
    return 1 if any(gpu in titulo.lower() for gpu in gpus) else 0

# --- Función Principal Exhaustiva ---
def scrapear_catalogo_exhaustivo():
    laptops_lista = []
    urls_vistas = set()
    idx = 0

    # 1. Cargar historial para reanudar el estado anterior
    if os.path.exists(ARCHIVO_CSV):
        try:
            df_previo = pd.read_csv(ARCHIVO_CSV)
            laptops_lista = df_previo.to_dict(orient="records")
            urls_vistas = set(df_previo["URL_Producto"].dropna().tolist())
            idx = len(laptops_lista)
            print(f"[i] Base de datos detectada: Se cargaron {idx} registros acumulados.")
            print(f"[i] Se omitirán {len(urls_vistas)} URLs procesadas anteriormente.\n")
        except Exception as e:
            print(f"[!] Ocurrió un inconveniente al leer la base previa ({e}). Se iniciará un catálogo nuevo.\n")

    # 2. Catálogo extendido de búsquedas para cobertura masiva
    segmentos = [
        # Marcas
        "laptop-hp", "laptop-lenovo", "laptop-dell", "laptop-asus", "laptop-acer",
        "macbook", "laptop-huawei", "laptop-msi", "laptop-samsung", "laptop-gigabyte",
        "laptop-alienware", "laptop-honor", "laptop-gateway", "laptop-ghia",

        # Tipos y Categorías
        "laptop-gamer", "notebook", "laptop-touch", "laptop-2-en-1", "ultrabook",
        "chromebook", "laptop-reacondicionada",

        # Procesadores
        "laptop-intel-core-i3", "laptop-intel-core-i5", "laptop-intel-core-i7", "laptop-intel-core-i9",
        "laptop-ryzen-3", "laptop-ryzen-5", "laptop-ryzen-7", "laptop-ryzen-9",
        "laptop-celeron", "laptop-athlon",

        # Especificaciones clave
        "laptop-8gb-ram", "laptop-16gb-ram", "laptop-32gb-ram", 
        "laptop-512gb-ssd", "laptop-1tb-ssd", "laptop-rtx"
    ]

    print(f"--- Iniciando extracción exhaustiva | Segmentos configurados: {len(segmentos)} ---\n")

    try:
        for num_seg, slug in enumerate(segmentos, 1):
            url_base = f"https://listado.mercadolibre.com.mx/{slug}"
            print(f"\n=== [Segmento {num_seg}/{len(segmentos)}] Procesando: '{slug}' | Registros totales: {idx} ===")
            
            pagina = 0
            paginas_sin_nuevos = 0

            while True:
                if pagina == 0:
                    url_actual = url_base
                else:
                    offset = (pagina * 48) + 1
                    url_actual = f"https://listado.mercadolibre.com.mx/{slug}_Desde_{offset}_NoIndex_True"

                print(f"  -> Escaneando página {pagina + 1} | Total acumulado: {idx}")
                
                try:
                    response = requests.get(url_actual, headers=HEADERS, timeout=20)
                except Exception as err:
                    print(f"  [!] Fallo de red en página {pagina + 1}: {err}. Pasando al siguiente segmento.")
                    break

                if response.status_code != 200 or "ingresa a tu cuenta" in response.text.lower():
                    print(f"  [!] Límite de paginación alcanzado o código HTTP {response.status_code} para '{slug}'.")
                    break

                soup = BeautifulSoup(response.text, "html.parser")
                tarjetas = soup.find_all(["div", "li"], class_=re.compile(r"poly-card|ui-search-layout__item"))

                if not tarjetas:
                    print(f"  [!] Fin de resultados para '{slug}'.")
                    break

                nuevos_en_esta_pagina = 0

                for item in tarjetas:
                    titulo_elem = item.find("a", class_=re.compile(r"poly-component__title|ui-search-item__title")) or item.find(re.compile(r"h[2-3]"))
                    titulo = titulo_elem.text.strip() if titulo_elem else ""

                    if not titulo or titulo == "Sin título" or "mac mini" in titulo.lower():
                        continue

                    url_producto = titulo_elem["href"] if titulo_elem and titulo_elem.has_attr("href") else ""
                    
                    # Ignorar productos ya registrados en corridas previas o segmentos previos
                    if not url_producto or url_producto in urls_vistas:
                        continue
                    
                    urls_vistas.add(url_producto)

                    precio_elem = item.find("span", class_="andes-money-amount__fraction")
                    precio_num = limpiar_precio(precio_elem.text.strip()) if precio_elem else None

                    if not precio_num or precio_num == 0:
                        continue

                    # Descarga de imagen
                    img_elem = item.find("img")
                    img_url = img_elem.get("data-src") or img_elem.get("src") or "" if img_elem else ""

                    ruta_img_local = ""
                    if img_url and img_url.startswith("http"):
                        ruta_img_local = os.path.join(CARPETA_IMAGENES, f"laptop_{idx:04d}.jpg")
                        try:
                            img_data = requests.get(img_url, headers=HEADERS, timeout=5).content
                            with open(ruta_img_local, "wb") as f:
                                f.write(img_data)
                        except Exception:
                            ruta_img_local = ""

                    alm_cap, alm_tipo = extraer_almacenamiento(titulo)

                    laptops_lista.append({
                        "ID_Producto": f"MLM-{idx:04d}",
                        "Nombre": titulo,
                        "Marca": extraer_marca(titulo),
                        "Precio_MXN": precio_num,
                        "RAM_GB": extraer_ram(titulo),
                        "Almacenamiento_GB": alm_cap,
                        "Tipo_Almacenamiento": alm_tipo,
                        "Procesador": extraer_procesador(titulo),
                        "Pantalla_Pulgadas": extraer_pantalla(titulo),
                        "GPU_Dedicada": detectar_gpu_dedicada(titulo),
                        "Ruta_Imagen_Local": ruta_img_local,
                        "URL_Producto": url_producto,
                    })
                    
                    idx += 1
                    nuevos_en_esta_pagina += 1

                if nuevos_en_esta_pagina == 0:
                    paginas_sin_nuevos += 1
                    if paginas_sin_nuevos >= 2:
                        print(f"  [!] Sin items nuevos tras 2 páginas consecutivas. Finalizando segmento '{slug}'.")
                        break
                else:
                    paginas_sin_nuevos = 0

                # Resguardo cada 250 nuevos registros
                if idx > 0 and idx % 250 == 0:
                    pd.DataFrame(laptops_lista).to_csv(ARCHIVO_CSV, index=False, encoding='utf-8')
                    print(f"   --> [Autoguardado] {idx} registros consolidados en '{ARCHIVO_CSV}'.")

                pagina += 1
                time.sleep(random.uniform(1.4, 2.4))

    except KeyboardInterrupt:
        print("\n[!] Proceso pausado manualmente por el usuario.")
    except Exception as e:
        print(f"\n[!] Excepción general: {e}")
    finally:
        if laptops_lista:
            df_final = pd.DataFrame(laptops_lista)
            df_final.to_csv(ARCHIVO_CSV, index=False, encoding='utf-8')
            print(f"\n¡Extracción finalizada! Se guardó un total de {len(df_final)} registros en '{ARCHIVO_CSV}'.")
            return df_final
        else:
            print("\nNo hay datos cargados.")
            return pd.DataFrame()

if __name__ == "__main__":
    df = scrapear_catalogo_exhaustivo()

[i] Base de datos detectada: Se cargaron 10000 registros acumulados.
[i] Se omitirán 10000 URLs procesadas anteriormente.

--- Iniciando extracción exhaustiva | Segmentos configurados: 37 ---


=== [Segmento 1/37] Procesando: 'laptop-hp' | Registros totales: 10000 ===
  -> Escaneando página 1 | Total acumulado: 10000
  -> Escaneando página 2 | Total acumulado: 10048
  -> Escaneando página 3 | Total acumulado: 10096
  -> Escaneando página 4 | Total acumulado: 10144
  -> Escaneando página 5 | Total acumulado: 10192
  -> Escaneando página 6 | Total acumulado: 10240
  -> Escaneando página 7 | Total acumulado: 10288
  -> Escaneando página 8 | Total acumulado: 10336
  -> Escaneando página 9 | Total acumulado: 10384
  -> Escaneando página 10 | Total acumulado: 10432
  -> Escaneando página 11 | Total acumulado: 10480
  -> Escaneando página 12 | Total acumulado: 10528
  -> Escaneando página 13 | Total acumulado: 10576
  -> Escaneando página 14 | Total acumulado: 10624
  -> Escaneando página 15 

---

## Consolidación de tablas

In [2]:
import pandas as pd

# 1. Cargar las 3 bases de datos
df_che = pd.read_csv("laptops_chedraui.csv")
df_ekt = pd.read_csv("laptops_elektra.csv")
df_ml = pd.read_csv("/media/solaris/017116de-6c2d-44fb-a97a-0afccc5d6037/solaris/claralap/laptops_mercadolibre.csv")

# 2. Asignar/Asegurar la columna de identificación de la tienda
df_che["Tienda"] = "Chedraui"
df_ekt["Tienda"] = "Elektra"
if "Tienda" not in df_ml.columns:
    df_ml["Tienda"] = "Mercado Libre"

# 3. Concatenar los 3 DataFrames
df_consolidado = pd.concat([df_che, df_ekt, df_ml], ignore_index=True)

# 4. Limpiar duplicados basados en la URL del producto
df_consolidado.drop_duplicates(subset=["URL_Producto"], keep="first", inplace=True)

# 5. Reordenar índice y regenerar ID único global
df_consolidado.reset_index(drop=True, inplace=True)
df_consolidado["ID_Global"] = [f"MX-{i:05d}" for i in range(len(df_consolidado))]

# Reorganizar columnas para poner 'ID_Global' y 'Tienda' al principio
cols = ["ID_Global", "Tienda"] + [c for c in df_consolidado.columns if c not in ["ID_Global", "Tienda"]]
df_consolidado = df_consolidado[cols]

# 6. Guardar el archivo consolidado final
archivo_salida = "dataset_laptops_mexico.csv"
df_consolidado.to_csv(archivo_salida, index=False, encoding="utf-8")

# 7. Resumen de registros por tienda
print(f"\n¡Dataset consolidado con éxito en '{archivo_salida}'!")
print(f"Total de registros unificados: {len(df_consolidado)}\n")
print("Desglose por tienda:")
print(df_consolidado["Tienda"].value_counts())


¡Dataset consolidado con éxito en 'dataset_laptops_mexico.csv'!
Total de registros unificados: 35412

Desglose por tienda:
Tienda
Mercado Libre    34306
Elektra           1000
Chedraui           106
Name: count, dtype: int64
